### Steps
```
1. merge tables
2. keep cols only available in test(kaggle)
3. lowercase cols
4. drop when store closed
5. delete columns: open
6. school holiday: convert to int
7. state holiday: convert to str
8. extend date
```

# features
```
class FeatureEngineering:   # before split
class FeatureTransformer:   # after split (fit only on train)
class ModelTrainer
```

In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))
pd.set_option("display.max_columns", None)

from src.services.preprocessing import FeatureEngineering, FeatureTransformer

In [2]:
df_rossmann = pd.read_csv("../data/Rossmann Stores Data.csv")
df_store = pd.read_csv("../data/store.csv")
df_rossmann.merge(df_store, how="left", on="Store").head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [3]:
feature_eng = FeatureEngineering()
df = feature_eng.merge_and_transform_dfs(df_rossmann=df_rossmann, df_store=df_store, is_train=True)
df = feature_eng.filter_open_stores(df=df, cols2drop=None)
df = feature_eng.data_preprocessing(df=df)

shape rossmann: (1017209, 9)
shape store: (1115, 10)
shape after merge: (1017209, 18)
shape with test cols only: (1017209, 8)
converting columns to lowercase...
shape: (1017209, 8)
shape: (844392, 8)
shape: (844392, 7)
shape: (844392, 7)
shape: (844392, 9)


In [4]:
# X, y = df.drop("sales", axis=1), df["sales"]
size = int(df.shape[0]*0.8)
train = df.iloc[:size]
test = df.iloc[size:]

In [5]:
feature_trans = FeatureTransformer()
feature_trans.fit(train.copy())
train_trans = feature_trans.transform(train)
test_trans = feature_trans.transform(test)

In [6]:
X_train = train_trans.drop("sales", axis=1)
y_train = train_trans["sales"]

In [7]:
X_test = test_trans.drop("sales", axis=1)
y_test = test_trans["sales"]

In [8]:
model = DecisionTreeRegressor(max_depth=20,random_state=0)
model.fit(X_train, y_train)


train_prediction = model.predict(X_train)
test_prediction = model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_prediction)
test_mae = mean_absolute_error(y_test, test_prediction)

print(f"train mae: {train_mae}")
print(f"test mae : {test_mae}")

train mae: 1589.4353959225368
test mae : 1867.6761554745046
